## Scripted control through TCP/JSON
FcsIT exposes registered GUI and analysis operations through a local, newline-delimited JSON interface. Start the application with `python FcsIT.py --tcp-server`; it listens on `127.0.0.1:8765` by default. Use `--tcp-port` and `--tcp-timeout` to change the port and command timeout. Set `FCSIT_TCP_TOKEN` before launch to require the same token in every request.

The installed command-line client accepts a command name followed by one argument object. Shell quoting differs between Linux and Windows.

#### Linux (Bash)

```bash
fcsit_call system.ping
fcsit_call gui.select_method '{"name":"fitting"}'
```

Bash preserves the double quotes inside a JSON object enclosed in single quotes.

#### Windows (Command Prompt)

```bat
fcsit_call system.ping
fcsit_call gui.select_method "{'name':'fitting'}"
```

The Windows client accepts this single-quoted object notation and converts it to standard JSON before sending the request. This compatibility syntax is intended for `cmd.exe`; the TCP protocol itself always uses valid JSON.

The runtime registry is authoritative. List it or inspect the argument schema and command metadata with:

Linux (Bash):

```bash
fcsit_call system.list_commands
fcsit_call system.describe_command '{"command":"fitting.fit_current"}'
```

Windows (Command Prompt):

```bat
fcsit_call system.list_commands
fcsit_call system.describe_command "{'command':'fitting.fit_current'}"
```

The server accepts only registered commands, binds only to localhost, and does not expose arbitrary Python or shell execution. Commands marked as destructive remove current session data or generated analysis state. Export important results before using them.

### System and GUI commands

- `system.ping` — check whether FcsIT and the TCP/JSON protocol are ready.
- `system.list_commands` — return the complete, sorted runtime command whitelist.
- `system.describe_command` — return arguments, constraints, return metadata, and risk flags for one command.
- `gui.get_state` — read the active module, available modules, and viewport state.
- `gui.list_commands` — deprecated compatibility alias for `system.list_commands`.
- `gui.select_method` — select `fitting`, `ptu_corr`, or `time_bin_corr`.
- `gui.set_settings_visible` — show or hide the Settings window.
- `gui.set_theme` — select the `dark` or `light` theme for the next application start.
- `gui.set_plot_export_formats` — enable or disable CSV and Pickle plot exports.
- `gui.save_settings` — save the current application settings as defaults.

### FCS fitting commands

Select the module first with `gui.select_method` and `name` set to `fitting`.

- `fitting.load_directory` — load binary, two-column, three-column, or multicolumn correlation data.
- `fitting.select_file` — select one exact filename from the loaded curves.
- `fitting.list_models` — list analytical models and the currently selected model.
- `fitting.set_model` — select an analytical model by its exact registered name.
- `fitting.set_parameter` — set a model parameter value, bounds, and fixed/free state.
- `fitting.get_parameter_controls` — read all controls for the active model parameters.
- `fitting.fit_current` — fit the currently selected curve.
- `fitting.fit_all` — fit all loaded curves and store their results.
- `fitting.set_options` — set weighting and the fitted lag-time range.
- `fitting.reset_tau_range` — restore the complete lag-time range of the loaded curve.
- `fitting.set_time_units` — set the source lag-time unit multiplier in seconds.
- `fitting.set_correlation_units` — set the source `G(tau)` multiplier.
- `fitting.get_diagnostics` — read fit statistics, residual diagnostics, and registered errors.
- `fitting.keep_current` — add the current fitted curve to the results table.
- `fitting.show_results` — display and return the stored results table.
- `fitting.close_results` — close the results window without removing records.
- `fitting.mark_result_for_removal` — select or deselect one stored row for removal.
- `fitting.remove_marked_results` — remove all marked result rows (destructive).
- `fitting.remove_result` — remove stored results for one source filename (destructive).
- `fitting.reset_workspace` — clear per-file settings and stored results (destructive).
- `fitting.reset_results` — clear results while preserving per-file settings (destructive).
- `fitting.export_results` — export stored results as CSV, DAT, or Pickle.
- `fitting.plot_all` — export plots for all stored fits.
- `fitting.get_results` — read current parameters, errors, fit statistics, and stored records.

### PTU/PT3 correlation commands

Select the module first with `gui.select_method` and `name` set to `ptu_corr`. Calculating filters is required before correlation, including workflows with time gating and background correction disabled.

- `ptu_corr.load_directory` — load supported PTU and PT3 measurements from a directory.
- `ptu_corr.select_file` — select one loaded PTU or PT3 filename.
- `ptu_corr.forget_current_measurement` — remove the selected measurement session (destructive).
- `ptu_corr.forget_all_measurements` — remove sessions for all loaded measurements (destructive).
- `ptu_corr.set_parameters` — set bin width, correlation points, chunks, lag range, cross-correlation, time gating, and background correction.
- `ptu_corr.set_filtering` — set TCSPC gates and channel background levels.
- `ptu_corr.set_custom_chunks` — define ordered, non-overlapping time-trace ranges.
- `ptu_corr.select_tab` — select the `tcspc` or `correlation` analysis tab.
- `ptu_corr.calculate_filters_current` — calculate filters for the selected measurement.
- `ptu_corr.calculate_filters_all` — calculate filters for every loaded measurement.
- `ptu_corr.correlate_current` — correlate the selected measurement.
- `ptu_corr.correlate_all` — correlate all loaded measurements.
- `ptu_corr.export_current` — export the selected measurement as CORR or DAT.
- `ptu_corr.export_all` — export every calculated measurement as CORR or DAT.
- `ptu_corr.get_state` — read files, channels, parameters, filtering, and chunk state.

### Time-binned correlation commands

Select the module first with `gui.select_method` and `name` set to `time_bin_corr`.

- `time_bin_corr.load_directory` — load time-binned inputs and optionally select an output directory.
- `time_bin_corr.select_file` — select one loaded measurement.
- `time_bin_corr.forget_current_measurement` — remove the selected session and its generated outputs (destructive).
- `time_bin_corr.forget_all_measurements` — remove all loaded sessions and outputs (destructive).
- `time_bin_corr.set_parameters` — set bin width, correlation points, chunks, lag range, and cross-correlation.
- `time_bin_corr.set_custom_chunks` — define ordered, non-overlapping signal ranges.
- `time_bin_corr.correlate_current` — correlate and automatically export the selected measurement.
- `time_bin_corr.correlate_all` — correlate and automatically export all measurements.
- `time_bin_corr.get_state` — read files, selected measurement, parameters, chunks, and generated outputs.

### Raw TCP/JSON request

Each request is one UTF-8 JSON object terminated by a newline. The `arguments` field is always an object, even when it is empty:

```json
{"protocol":"fcsit-tcp-json","version":"1.0","id":"request-1","command":"system.ping","arguments":{}}
```

A successful synchronous response has `status` equal to `completed`. An `error` response contains an explanatory `error` field. Long-running analysis commands can also return `accepted` when the configured TCP timeout expires; use state and output commands to verify completion before starting dependent operations.